# Theorem 9 — paired representation-risk bound

**Formal source:** [`../09_unified_representation_risk_bound.md`](../09_unified_representation_risk_bound.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
rng = np.random.default_rng(9)
target = rng.normal(size=20000)
ideal = target.copy()
approximate = ideal + rng.normal(0, 0.15, target.size)
risk_gap = np.mean(abs(approximate - target)) - np.mean(abs(ideal - target))
paired_error = np.mean(abs(approximate - ideal))
assert abs(risk_gap) <= paired_error + 1e-12
binary = np.tile([0.0, 1.0], 5000)
good, reversed_semantics = binary, 1 - binary
assert np.mean(abs(np.sort(good) - np.sort(reversed_semantics))) == 0
assert np.mean(abs(good - binary)) == 0 and np.mean(abs(reversed_semantics - binary)) == 1
print({"paired_risk_gap": float(risk_gap), "paired_error": float(paired_error), "semantic_reversal_risk": 1.0})

In [ ]:
print('THEORY_DEMO_PASS::09_unified_representation_risk_bound')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')